# Week 4 Assignment: Object Detection and Segmentation
## AI in Computer Vision — Pascal VOC + YOLO + U-Net

**Scenario:** We are building a smart surveillance system that detects and segments objects in images. We use:
- **YOLO** (You Only Look Once) to detect objects and draw bounding boxes
- **U-Net** to segment (outline) the detected regions
- **Pascal VOC 2012** as our training dataset

This notebook walks through every step: environment setup → data loading → U-Net training → YOLO detection → combined inference.

---
## Step 1: Install and Import Libraries

We install the packages we need and then import them. `ultralytics` gives us YOLO, `torchvision` gives us Pascal VOC, and `torch` is the deep learning framework for U-Net.

In [ ]:
# Install required packages (run once in Colab)
!pip install ultralytics torch torchvision opencv-python numpy matplotlib pillow --quiet

In [ ]:
import os
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from PIL import Image
from ultralytics import YOLO

# Create outputs folder to save results
os.makedirs('outputs', exist_ok=True)

# Use GPU if available, otherwise CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

---
## Step 2: Load and Preprocess the Pascal VOC 2012 Dataset

Pascal VOC 2012 contains 20 object categories including people, cars, animals, and more. Each image has a matching segmentation mask where each pixel is labeled with a class.

We:
- Resize all images to **128×128** so training is faster
- Normalize pixel values to the standard ImageNet mean and std
- Use only a **small subset (200 images)** so this runs in a reasonable time in Colab
- Convert masks to class index tensors (not normalized floats)

In [ ]:
IMG_SIZE = 128  # resize target for both images and masks
BATCH_SIZE = 8
SUBSET_SIZE = 200  # use a small slice so training finishes quickly

# Image transform: resize → tensor → normalize
image_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Mask transform: resize with NEAREST to preserve integer class labels
# We keep it as a PIL image first; we convert to a long tensor manually below
mask_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=transforms.InterpolationMode.NEAREST),
])

In [ ]:
# Custom dataset wrapper to handle mask conversion correctly
class VOCSegDataset(torch.utils.data.Dataset):
    """Wraps VOCSegmentation to return (image_tensor, mask_tensor) pairs."""

    def __init__(self, root, year, image_set, download):
        self.base = datasets.VOCSegmentation(
            root=root,
            year=year,
            image_set=image_set,
            download=download
        )

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img_pil, mask_pil = self.base[idx]  # both are PIL images

        # Apply image transforms (resize + normalize)
        img_tensor = image_transform(img_pil)

        # Resize mask with NEAREST interpolation to keep integer labels
        mask_pil = mask_transform(mask_pil)

        # Convert mask to numpy, clamp boundary/ignore pixels (255) to 0
        mask_np = np.array(mask_pil, dtype=np.int64)
        mask_np[mask_np == 255] = 0  # VOC uses 255 for boundary/ignore

        mask_tensor = torch.from_numpy(mask_np).long()  # shape: (H, W)
        return img_tensor, mask_tensor


# Store dataset in the user's home directory so it works on any machine
DATA_ROOT = os.path.join(os.path.expanduser('~'), 'voc_data')
os.makedirs(DATA_ROOT, exist_ok=True)

print('Loading Pascal VOC 2012 dataset (this may take a minute to download)...')
full_dataset = VOCSegDataset(root=DATA_ROOT, year='2012', image_set='train', download=True)

# Take a small subset for faster training
indices = list(range(min(SUBSET_SIZE, len(full_dataset))))
subset = Subset(full_dataset, indices)

# num_workers=0 is required on Windows to avoid DataLoader multiprocessing issues
dataloader = DataLoader(subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
print(f'Dataset loaded. Using {len(subset)} images, {len(dataloader)} batches.')

In [ ]:
# Visualize a few samples from the dataset
VOC_CLASSES = [
    'background', 'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow', 'diningtable', 'dog', 'horse',
    'motorbike', 'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
]

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
mean = np.array([0.485, 0.456, 0.406])
std  = np.array([0.229, 0.224, 0.225])

for i in range(4):
    img_t, mask_t = full_dataset[i]
    # Denormalize for display
    img_disp = img_t.numpy().transpose(1, 2, 0) * std + mean
    img_disp = np.clip(img_disp, 0, 1)

    axes[0, i].imshow(img_disp)
    axes[0, i].set_title(f'Image {i}')
    axes[0, i].axis('off')

    axes[1, i].imshow(mask_t.numpy(), cmap='tab20', vmin=0, vmax=20)
    axes[1, i].set_title(f'Mask {i}')
    axes[1, i].axis('off')

plt.suptitle('Pascal VOC 2012 — Sample Images and Segmentation Masks', fontsize=13)
plt.tight_layout()
plt.savefig('outputs/dataset_samples.png', dpi=100)
plt.show()
print('Saved: outputs/dataset_samples.png')

---
## Step 3: Define the U-Net Architecture

U-Net is a convolutional neural network designed for image segmentation. Its key features are:
- **Encoder (contracting path):** progressively reduces spatial size while increasing feature channels
- **Decoder (expanding path):** upsamples back to the original size
- **Skip connections:** copy features from encoder to decoder so fine details aren't lost
- **Output layer:** 1×1 convolution that maps to the number of classes (21 for VOC)

We use a simplified U-Net with 3 encoder levels to keep training fast.

In [ ]:
NUM_CLASSES = 21  # 20 VOC object classes + 1 background

class DoubleConv(nn.Module):
    """Two consecutive Conv→BN→ReLU blocks — the basic building block of U-Net."""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    """Simplified U-Net for semantic segmentation."""

    def __init__(self, in_channels=3, num_classes=NUM_CLASSES):
        super().__init__()

        # ---- Encoder ----
        self.enc1 = DoubleConv(in_channels, 64)   # 128×128 → 128×128
        self.enc2 = DoubleConv(64, 128)            # 64×64   → 64×64
        self.enc3 = DoubleConv(128, 256)           # 32×32   → 32×32

        self.pool = nn.MaxPool2d(2)                # halves spatial dims

        # ---- Bottleneck ----
        self.bottleneck = DoubleConv(256, 512)     # 16×16

        # ---- Decoder ----
        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(512, 256)           # 512 = 256 skip + 256 up

        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(256, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(128, 64)

        # ---- Output ----
        self.out_conv = nn.Conv2d(64, num_classes, kernel_size=1)

    def forward(self, x):
        # Encoder — save each level for skip connections
        e1 = self.enc1(x)               # (B, 64,  H,   W)
        e2 = self.enc2(self.pool(e1))   # (B, 128, H/2, W/2)
        e3 = self.enc3(self.pool(e2))   # (B, 256, H/4, W/4)

        # Bottleneck
        b  = self.bottleneck(self.pool(e3))  # (B, 512, H/8, W/8)

        # Decoder — upsample then concatenate the matching encoder output
        d3 = self.dec3(torch.cat([self.up3(b),  e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        return self.out_conv(d1)  # (B, NUM_CLASSES, H, W)


unet_model = UNet(in_channels=3, num_classes=NUM_CLASSES).to(device)

# Count parameters
total_params = sum(p.numel() for p in unet_model.parameters())
print(f'U-Net created. Total parameters: {total_params:,}')
print(unet_model)

---
## Step 4: Train the U-Net Model

We train U-Net to predict a class label for every pixel in an image. The loss function is **Cross-Entropy Loss**, which penalizes wrong class predictions. We use the **Adam** optimizer to update weights.

Training for just **5 epochs** on 200 images is enough to demonstrate the pipeline — a production model would need many more epochs and data.

In [ ]:
NUM_EPOCHS = 5
LEARNING_RATE = 1e-3

# ignore_index=255 tells the loss to skip any remaining boundary pixels
criterion = nn.CrossEntropyLoss(ignore_index=255)
optimizer = optim.Adam(unet_model.parameters(), lr=LEARNING_RATE)

epoch_losses = []
num_batches = len(dataloader)

print('Starting U-Net training...')
print(f'  {NUM_EPOCHS} epochs x {num_batches} batches = {NUM_EPOCHS * num_batches} total steps')
print('-' * 50)

for epoch in range(1, NUM_EPOCHS + 1):
    unet_model.train()
    running_loss = 0.0

    for batch_idx, (images, masks) in enumerate(dataloader):
        images = images.to(device)
        masks  = masks.to(device)

        outputs = unet_model(images)
        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        if (batch_idx + 1) % 5 == 0 or (batch_idx + 1) == num_batches:
            print(f'  Epoch {epoch}/{NUM_EPOCHS}  Batch {batch_idx+1}/{num_batches}  Loss: {loss.item():.4f}')

    avg_loss = running_loss / num_batches
    epoch_losses.append(avg_loss)
    print(f'Epoch [{epoch}/{NUM_EPOCHS}] complete  Avg Loss: {avg_loss:.4f}')
    print('-' * 50)

print('Training complete.')

In [ ]:
# Plot and save the training loss curve
plt.figure(figsize=(8, 5))
plt.plot(range(1, NUM_EPOCHS + 1), epoch_losses, marker='o', linewidth=2, color='steelblue')
plt.title('U-Net Training Loss per Epoch', fontsize=14)
plt.xlabel('Epoch')
plt.ylabel('Cross-Entropy Loss')
plt.xticks(range(1, NUM_EPOCHS + 1))
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('outputs/training_loss.png', dpi=100)
plt.show()
print('Saved: outputs/training_loss.png')

---
## Step 5: Save and Test the U-Net on Validation Images

Let's run the trained U-Net on a few images from the dataset to see what segmentation masks it predicts. Since we only trained briefly, the masks will be rough but should show the model is learning.

In [ ]:
# Save the trained U-Net weights
torch.save(unet_model.state_dict(), 'outputs/unet_weights.pth')
print('U-Net weights saved to outputs/unet_weights.pth')

# Run inference on a few dataset images
unet_model.eval()

fig, axes = plt.subplots(3, 3, figsize=(12, 10))
mean = np.array([0.485, 0.456, 0.406])
std  = np.array([0.229, 0.224, 0.225])

with torch.no_grad():
    for col in range(3):
        img_t, mask_t = full_dataset[col]

        # Predict
        pred = unet_model(img_t.unsqueeze(0).to(device))  # (1, C, H, W)
        pred_mask = pred.argmax(dim=1).squeeze(0).cpu().numpy()  # (H, W)

        # Denormalize image for display
        img_disp = img_t.numpy().transpose(1, 2, 0) * std + mean
        img_disp = np.clip(img_disp, 0, 1)

        axes[0, col].imshow(img_disp);             axes[0, col].set_title(f'Input Image {col}');        axes[0, col].axis('off')
        axes[1, col].imshow(mask_t.numpy(), cmap='tab20', vmin=0, vmax=20); axes[1, col].set_title('Ground Truth Mask');  axes[1, col].axis('off')
        axes[2, col].imshow(pred_mask, cmap='tab20', vmin=0, vmax=20);      axes[2, col].set_title('U-Net Predicted Mask'); axes[2, col].axis('off')

plt.suptitle('U-Net Segmentation Results', fontsize=13)
plt.tight_layout()
plt.savefig('outputs/unet_predictions.png', dpi=100)
plt.show()
print('Saved: outputs/unet_predictions.png')

---
## Step 6: YOLO Object Detection

YOLO (You Only Look Once) is a real-time object detection model. We load the pre-trained **YOLOv8n** (nano) model — the lightest version, perfect for demonstrating detection quickly.

YOLO returns:
- **Bounding boxes** — the coordinates of a rectangle around each detected object
- **Class labels** — what object was detected (e.g., person, car, dog)
- **Confidence scores** — how sure the model is about each detection

We run it on a few Pascal VOC images.

In [ ]:
# Load pre-trained YOLOv8n (downloads automatically on first run)
yolo_model = YOLO('yolov8n.pt')
print('YOLOv8n loaded successfully.')

In [ ]:
# Helper: reverse the image normalization so we can display/pass to YOLO
def denormalize(img_tensor):
    """Convert a normalized tensor back to a uint8 NumPy array (H, W, 3)."""
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img = img_tensor.numpy().transpose(1, 2, 0) * std + mean
    img = np.clip(img, 0, 1)
    return np.ascontiguousarray((img * 255).astype(np.uint8))


# Run YOLO on 4 sample images and display results
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for i in range(4):
    img_t, _ = full_dataset[i]
    img_np = denormalize(img_t)  # uint8 RGB

    # YOLO expects BGR for cv2, but ultralytics handles RGB arrays fine
    results = yolo_model(img_np, verbose=False)

    # Draw boxes using ultralytics built-in plot()
    annotated = results[0].plot()          # returns BGR uint8
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

    axes[i].imshow(annotated_rgb)
    axes[i].set_title(f'YOLO Detection — Image {i}')
    axes[i].axis('off')

    # Save each YOLO result individually
    cv2.imwrite(f'outputs/yolo_detection_{i}.png', annotated)

plt.suptitle('YOLOv8 Object Detection on Pascal VOC Images', fontsize=13)
plt.tight_layout()
plt.savefig('outputs/yolo_detections_grid.png', dpi=100)
plt.show()
print('Saved: outputs/yolo_detection_*.png and outputs/yolo_detections_grid.png')

---
## Step 7: Combined YOLO + U-Net Inference Pipeline

This is the core of our smart surveillance system. The pipeline works like this:

1. **YOLO detects objects** and returns bounding boxes
2. **We crop each detected region** from the original image
3. **We resize the crop** to 128×128 for U-Net
4. **U-Net segments the crop** — classifying every pixel
5. **We display and save** the original, YOLO output, crop, mask, and overlay

This lets us first find *where* objects are, then precisely outline *what* they look like.

In [ ]:
def run_combined_pipeline(dataset, sample_idx=0):
    """
    Full detection + segmentation pipeline for one image.
    Returns a figure showing all intermediate outputs.
    """
    img_t, gt_mask = dataset[sample_idx]
    img_np = denormalize(img_t)  # (H, W, 3) uint8 RGB

    # --- Step A: YOLO detection ---
    results = yolo_model(img_np, verbose=False)
    yolo_annotated = results[0].plot()                          # BGR
    yolo_annotated_rgb = cv2.cvtColor(yolo_annotated, cv2.COLOR_BGR2RGB)

    # --- Step B: Extract bounding boxes ---
    boxes = results[0].boxes
    H, W = img_np.shape[:2]

    if boxes is not None and len(boxes) > 0:
        # Use the highest-confidence detection
        best_idx = boxes.conf.argmax().item()
        x1, y1, x2, y2 = boxes.xyxy[best_idx].cpu().numpy().astype(int)
        # Clamp to image bounds
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(W, x2), min(H, y2)
        label = results[0].names[int(boxes.cls[best_idx].item())]
        conf  = float(boxes.conf[best_idx].item())
        print(f'Best detection: {label} ({conf:.2f}) at [{x1},{y1},{x2},{y2}]')
    else:
        # No detection — use the whole image
        x1, y1, x2, y2 = 0, 0, W, H
        label, conf = 'no detection', 0.0
        print('No YOLO detections — using full image for segmentation.')

    # --- Step C: Crop the detected region ---
    crop_np = img_np[y1:y2, x1:x2]  # (crop_H, crop_W, 3) uint8
    if crop_np.size == 0:            # safety check for degenerate boxes
        crop_np = img_np

    # --- Step D: Preprocess crop for U-Net ---
    crop_pil = Image.fromarray(crop_np)
    crop_tensor = image_transform(crop_pil).unsqueeze(0).to(device)  # (1,3,128,128)

    # --- Step E: U-Net segmentation ---
    unet_model.eval()
    with torch.no_grad():
        seg_output = unet_model(crop_tensor)                      # (1, C, 128, 128)
    pred_mask = seg_output.argmax(dim=1).squeeze(0).cpu().numpy() # (128, 128)

    # --- Step F: Create overlay (colour map on original image region) ---
    # Resize mask back to crop size for overlay
    pred_mask_resized = cv2.resize(
        pred_mask.astype(np.uint8),
        (crop_np.shape[1], crop_np.shape[0]),
        interpolation=cv2.INTER_NEAREST
    )

    # Map class indices to colours using matplotlib tab20 colormap
    cmap = plt.get_cmap('tab20')
    mask_rgb = (cmap(pred_mask_resized / 20.0)[:, :, :3] * 255).astype(np.uint8)
    overlay = cv2.addWeighted(crop_np, 0.6, mask_rgb, 0.4, 0)

    # Full-image overlay (paste back into image)
    full_overlay = img_np.copy()
    full_overlay[y1:y2, x1:x2] = overlay

    # --- Plot everything ---
    fig, axes = plt.subplots(1, 5, figsize=(22, 5))

    axes[0].imshow(img_np);              axes[0].set_title('1. Original Image');            axes[0].axis('off')
    axes[1].imshow(yolo_annotated_rgb);  axes[1].set_title(f'2. YOLO: {label} ({conf:.0%})'); axes[1].axis('off')
    axes[2].imshow(crop_np);             axes[2].set_title('3. Cropped Region');             axes[2].axis('off')
    axes[3].imshow(pred_mask, cmap='tab20', vmin=0, vmax=20); axes[3].set_title('4. U-Net Mask'); axes[3].axis('off')
    axes[4].imshow(full_overlay);        axes[4].set_title('5. Combined Result');            axes[4].axis('off')

    plt.suptitle('YOLO Detection → U-Net Segmentation Pipeline', fontsize=13)
    plt.tight_layout()

    return fig, img_np, yolo_annotated, pred_mask, full_overlay


# Run on sample image 0
fig, orig, yolo_out, mask_out, combined = run_combined_pipeline(full_dataset, sample_idx=0)
plt.savefig('outputs/combined_pipeline_0.png', dpi=100)
plt.show()
print('Saved: outputs/combined_pipeline_0.png')

In [ ]:
# Save the individual output components for sample 0
cv2.imwrite('outputs/original_image.png',   cv2.cvtColor(orig, cv2.COLOR_RGB2BGR))
cv2.imwrite('outputs/yolo_output.png',      yolo_out)          # already BGR
cv2.imwrite('outputs/combined_result.png',  cv2.cvtColor(combined, cv2.COLOR_RGB2BGR))

# Save U-Net mask as a coloured image
cmap = plt.get_cmap('tab20')
mask_colour = (cmap(mask_out / 20.0)[:, :, :3] * 255).astype(np.uint8)
cv2.imwrite('outputs/unet_mask.png', cv2.cvtColor(mask_colour, cv2.COLOR_RGB2BGR))

print('All individual outputs saved:')
print('  outputs/original_image.png')
print('  outputs/yolo_output.png')
print('  outputs/unet_mask.png')
print('  outputs/combined_result.png')

In [ ]:
# Run the pipeline on a second image for variety
fig2, _, _, _, _ = run_combined_pipeline(full_dataset, sample_idx=5)
plt.savefig('outputs/combined_pipeline_5.png', dpi=100)
plt.show()
print('Saved: outputs/combined_pipeline_5.png')

---
## Step 8: Summary of Results

Let's list all the output files we saved and print a brief summary of the training run.

In [ ]:
print('=' * 55)
print('  WEEK 4 ASSIGNMENT — RESULTS SUMMARY')
print('=' * 55)
print()
print('Dataset: Pascal VOC 2012')
print(f'  Training subset:  {len(subset)} images')
print(f'  Image size:       {IMG_SIZE}×{IMG_SIZE}')
print(f'  Classes:          {NUM_CLASSES} (20 VOC + background)')
print()
print('U-Net Training:')
for epoch, loss in enumerate(epoch_losses, 1):
    print(f'  Epoch {epoch}: loss = {loss:.4f}')
print()
print('YOLO Model: YOLOv8n (pre-trained on COCO)')
print()
print('Output Files:')
for f in sorted(os.listdir('outputs')):
    size_kb = os.path.getsize(f'outputs/{f}') // 1024
    print(f'  outputs/{f:40s}  {size_kb:5d} KB')
print()
print('Pipeline: YOLO → crop → U-Net → overlay complete.')
print('=' * 55)

---
## References

- Everingham, M., Van Gool, L., Williams, C. K. I., Winn, J., & Zisserman, A. (2010). The Pascal Visual Object Classes (VOC) challenge. *International Journal of Computer Vision, 88*(2), 303–338. https://doi.org/10.1007/s11263-009-0275-4
- Jocher, G., Chaurasia, A., & Qiu, J. (2023). *Ultralytics YOLO* (Version 8.0.0) [Software]. https://github.com/ultralytics/ultralytics
- Ronneberger, O., Fischer, P., & Brox, T. (2015). U-net: Convolutional networks for biomedical image segmentation. In *Lecture Notes in Computer Science* (pp. 234–241). Springer. https://doi.org/10.1007/978-3-319-24574-4_28
- PyTorch documentation. (n.d.). *torch.nn*. https://pytorch.org/docs/stable/nn.html
- Torchvision. (n.d.). *VOCSegmentation dataset*. https://pytorch.org/vision/stable/generated/torchvision.datasets.VOCSegmentation.html
- GeeksforGeeks. (2024, August 13). *Detect an object with OpenCV Python*. https://www.geeksforgeeks.org/detect-an-object-with-opencv-python/
- Lakshmanan, V., Görner, M., & Gillard, R. (n.d.). *Practical machine learning for computer vision*. O'Reilly Media. https://learning.oreilly.com/library/view/practical-machine-learning/9781098102357/ch04.html